# ML-10 — Content Action Playbook

**Applied Search Intelligence: Operationalizing ML Model Predictions into Ranked Human-in-the-Loop Actions**

A machine learning model nobody acts on is just a science project. This playbook bridges validated predictive scores and operational execution. It converts raw refresh probabilities into a ranked action queue enriched with human-trustworthy reason codes, clear operational limits, human-in-the-loop review criteria, explicit no-go automation boundaries, and quantitative retrain triggers.

## 1. Ranked actions + reason codes

*The queue: turning model scores into transparent, prioritized content refresh recommendations with human-interpretable reason codes.*

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
import json

# 1. Load anonymized dataset
data_path = Path('../../data/raw/content_refresh_anonymized.csv')
if not data_path.exists():
    data_path = Path('data/raw/content_refresh_anonymized.csv')

df = pd.read_csv(data_path)
print(f"Loaded dataset with {len(df):,} rows and {len(df.columns)} columns.")

# 2. Filter active scope: impressions > 0 and age >= 90 days
active_df = df[(df['impressions_90d'] > 0) & (df['content_age_days'] >= 90)].copy().reset_index(drop=True)
print(f"Active analysis dataset: {len(active_df):,} pages.")

# 3. Target definition & feature set
active_df['is_declining'] = (active_df['trend_direction'].str.lower() == 'down').astype(int)

feature_cols = [
    'search_volume', 'competition', 'cpc', 'word_count', 'char_count',
    'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d',
    'content_age_days', 'days_since_last_update', 'ctr', 'avg_position',
    'engagement_rate', 'scroll_rate', 'ai_traffic_pct'
]

X = active_df[feature_cols].fillna(0)
y = active_df['is_declining']

# 4. Fit Random Forest Model to compute calibrated refresh probabilities
rf = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42)
rf.fit(X, y)
active_df['refresh_score'] = rf.predict_proba(X)[:, 1]

# 5. Reason Code Rule Engine
def assign_reason_codes(row):
    codes = []
    # SERP Decay Candidate: Older content with significant drop
    if row['content_age_days'] >= 180 and row['impressions_last_30d'] < row['impressions_prev_30d'] * 0.7:
        codes.append('SERP_DECAY_CANDIDATE')
    
    # High CTR Low Position: Good relevance, positional ranking gap
    if row['ctr'] > 0.04 and row['avg_position'] > 10:
        codes.append('HIGH_CTR_LOW_POSITION')
    
    # Stagnant High Potential: High search volume & impressions but low clicks
    if row['impressions_90d'] > 5000 and row['clicks_90d'] < 100:
        codes.append('STAGNANT_HIGH_POTENTIAL')
    
    # Thin Content Decay: Low word count and declining
    if row['word_count'] < 600 and row['days_since_last_update'] > 200:
        codes.append('THIN_CONTENT_DECAY')
    
    # Low Engagement Risk
    if row['engagement_rate'] < 0.35 and row['sessions_90d'] > 200:
        codes.append('LOW_ENGAGEMENT_RISK')
        
    if not codes:
        codes.append('MODEL_SCORE_ELEVATED')
        
    return ' | '.join(codes)

def assign_action_type(row):
    if 'HIGH_CTR_LOW_POSITION' in row['reason_codes'] or 'STAGNANT_HIGH_POTENTIAL' in row['reason_codes']:
        return 'TITLE_META_RESTRUCTURE'
    elif 'THIN_CONTENT_DECAY' in row['reason_codes']:
        return 'EXPAND_CONTENT_DEPTH'
    elif 'SERP_DECAY_CANDIDATE' in row['reason_codes']:
        return 'COMPREHENSIVE_REFRESH'
    elif 'LOW_ENGAGEMENT_RISK' in row['reason_codes']:
        return 'UX_CRO_OPTIMIZATION'
    else:
        return 'EDITORIAL_REVIEW'

active_df['reason_codes'] = active_df.apply(assign_reason_codes, axis=1)
active_df['suggested_action'] = active_df.apply(assign_action_type, axis=1)

# Rank queue by refresh_score descending
queue_df = active_df.sort_values(by='refresh_score', ascending=False).reset_index(drop=True)
queue_df['priority_rank'] = queue_df.index + 1

# Add Impact Tier
def assign_impact_tier(row):
    if row['priority_rank'] <= 500:
        return 'P1_CRITICAL'
    elif row['priority_rank'] <= 2000:
        return 'P2_HIGH'
    elif row['priority_rank'] <= 5000:
        return 'P3_MEDIUM'
    else:
        return 'P4_LOW'

queue_df['impact_tier'] = queue_df.apply(assign_impact_tier, axis=1)

display_cols = ['priority_rank', 'content_id', 'client_id', 'refresh_score', 'impact_tier', 'suggested_action', 'reason_codes', 'impressions_90d', 'clicks_90d', 'avg_position']
print("\n--- Top 10 Ranked Action Queue Sample ---")
print(queue_df[display_cols].head(10).to_string(index=False))


Loaded dataset with 30,000 rows and 44 columns.
Active analysis dataset: 30,000 pages.

--- Top 10 Ranked Action Queue Sample ---
 priority_rank           content_id         client_id  refresh_score impact_tier       suggested_action                                                           reason_codes  impressions_90d  clicks_90d  avg_position
             1 content_875fc7912e9b client_7f2253d7e2       0.869156 P1_CRITICAL TITLE_META_RESTRUCTURE                           SERP_DECAY_CANDIDATE | HIGH_CTR_LOW_POSITION            21533         102          14.6
             2 content_d7e9829c3b39 client_7f2253d7e2       0.868544 P1_CRITICAL TITLE_META_RESTRUCTURE SERP_DECAY_CANDIDATE | HIGH_CTR_LOW_POSITION | STAGNANT_HIGH_POTENTIAL            13402          67          18.9
             3 content_41b9311ed497 client_7f2253d7e2       0.866809 P1_CRITICAL TITLE_META_RESTRUCTURE                        HIGH_CTR_LOW_POSITION | STAGNANT_HIGH_POTENTIAL            65669          64          35.

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

In [2]:
intended_use_spec = {
    "primary_users": [
        "FlyRank Content Editors & Strategists",
        "SEO Program Managers",
        "Digital Publishing Operations Teams"
    ],
    "operational_scope": [
        "Prioritizing monthly editorial content refresh queues",
        "Identifying decaying search traffic assets before core rank collapse",
        "Allocating copywriter resources to high-ROI existing content updates"
    ],
    "out_of_scope_and_limits": [
        "Brand and Navigational Queries: High brand-affinity pages must not be flagged solely for position drops.",
        "Seasonal Peak Content: E-commerce or holiday pages experiencing seasonal low-months are excluded from decay triggers.",
        "Newly Published Pages (< 90 Days): Content still in search indexing/sandbox phase is excluded.",
        "Causal Directives: The score flags priority for human inspection; it does not guarantee rank increase."
    ]
}

print("=== INTENDED USE & OPERATIONAL LIMITATIONS ===")
for key, values in intended_use_spec.items():
    print(f"\n[{key.upper().replace('_', ' ')}]")
    for item in values:
        print(f" - {item}")


=== INTENDED USE & OPERATIONAL LIMITATIONS ===

[PRIMARY USERS]
 - FlyRank Content Editors & Strategists
 - SEO Program Managers
 - Digital Publishing Operations Teams

[OPERATIONAL SCOPE]
 - Prioritizing monthly editorial content refresh queues
 - Identifying decaying search traffic assets before core rank collapse
 - Allocating copywriter resources to high-ROI existing content updates

[OUT OF SCOPE AND LIMITS]
 - Brand and Navigational Queries: High brand-affinity pages must not be flagged solely for position drops.
 - Seasonal Peak Content: E-commerce or holiday pages experiencing seasonal low-months are excluded from decay triggers.
 - Newly Published Pages (< 90 Days): Content still in search indexing/sandbox phase is excluded.
 - Causal Directives: The score flags priority for human inspection; it does not guarantee rank increase.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

In [3]:
review_protocol = {
    "human_review_workflow": [
        "Step 1: Editor inspects P1/P2 items in the ranked queue.",
        "Step 2: Cross-checks Reason Codes against live Search Console & SERP intent.",
        "Step 3: Confirms editorial quality, entity accuracy, and brand alignment.",
        "Step 4: Executes targeted refresh (e.g. update stats, rewrite intro, improve depth).",
        "Step 5: Logs refresh execution timestamp in FlyRank CMS."
    ],
    "no_go_automation_list": [
        "NO-GO 1: Automated Page Deletions or 301 Redirects without human sign-off.",
        "NO-GO 2: Auto-publishing AI-generated text directly to production without editorial review.",
        "NO-GO 3: Automatic modification of URL slugs or canonical tags.",
        "NO-GO 4: Altering legal, compliance, or medical/YMYL content without domain expert review."
    ],
    "cost_value_matrix": {
        "High Value / Low Effort": "Title & Meta Restructure (P1 queue items with HIGH_CTR_LOW_POSITION)",
        "High Value / High Effort": "Comprehensive Content Expansion (P1 items with SERP_DECAY_CANDIDATE)",
        "Low Value / High Effort": "Full rewrite of low-impression niche pages (Deferred / Backlog)"
    }
}

print("=== HUMAN REVIEW WORKFLOW & NO-GO BOUNDARIES ===")
print("\n--- Human Review Steps ---")
for step in review_protocol['human_review_workflow']:
    print(f"  {step}")

print("\n--- NO-GO AUTOMATION BOUNDARIES (STRICT) ---")
for nogo in review_protocol['no_go_automation_list']:
    print(f"  ⚠️ {nogo}")


=== HUMAN REVIEW WORKFLOW & NO-GO BOUNDARIES ===

--- Human Review Steps ---
  Step 1: Editor inspects P1/P2 items in the ranked queue.
  Step 2: Cross-checks Reason Codes against live Search Console & SERP intent.
  Step 3: Confirms editorial quality, entity accuracy, and brand alignment.
  Step 4: Executes targeted refresh (e.g. update stats, rewrite intro, improve depth).
  Step 5: Logs refresh execution timestamp in FlyRank CMS.

--- NO-GO AUTOMATION BOUNDARIES (STRICT) ---
  ⚠️ NO-GO 1: Automated Page Deletions or 301 Redirects without human sign-off.
  ⚠️ NO-GO 2: Auto-publishing AI-generated text directly to production without editorial review.
  ⚠️ NO-GO 3: Automatic modification of URL slugs or canonical tags.
  ⚠️ NO-GO 4: Altering legal, compliance, or medical/YMYL content without domain expert review.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

In [4]:
monitoring_config = {
    "metric_thresholds": {
        "precision_at_50_min": 0.60,
        "precision_at_100_min": 0.55,
        "lift_over_baseline_min": 2.0
    },
    "drift_indicators": [
        "Population Stability Index (PSI) > 0.25 on feature distributions (e.g. ctr, avg_position).",
        "Google Search Engine Layout Shift: Widespread introduction of SERP AI Overviews altering baseline CTRs.",
        "Refresh conversion degradation: P1 refresh actions yield < 40% rank recovery over 60 days."
    ],
    "retrain_schedule": {
        "cadence": "Monthly automated retrain pipeline execution.",
        "event_driven": "Trigger immediate retrain upon confirmed Google Core Algorithm Update release."
    }
}

print("=== MONITORING & RETRAIN TRIGGERS ===")
print(f"Minimum Precision@50 Threshold: {monitoring_config['metric_thresholds']['precision_at_50_min']:.2f}")
print(f"Minimum Lift over Baseline: {monitoring_config['metric_thresholds']['lift_over_baseline_min']:.1f}x")
print("\nDrift Detection Criteria:")
for indicator in monitoring_config['drift_indicators']:
    print(f" - {indicator}")


=== MONITORING & RETRAIN TRIGGERS ===
Minimum Precision@50 Threshold: 0.60
Minimum Lift over Baseline: 2.0x

Drift Detection Criteria:
 - Population Stability Index (PSI) > 0.25 on feature distributions (e.g. ctr, avg_position).
 - Google Search Engine Layout Shift: Widespread introduction of SERP AI Overviews altering baseline CTRs.
 - Refresh conversion degradation: P1 refresh actions yield < 40% rank recovery over 60 days.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [5]:
# Create output paths
out_dir = Path('../../work/outputs')
if not out_dir.parent.exists():
    out_dir = Path('work/outputs')
out_dir.mkdir(parents=True, exist_ok=True)

# Export Ranked Action Queue CSV
csv_export_path = out_dir / 'action_playbook_queue.csv'
queue_df.to_csv(csv_export_path, index=False)
print(f"Wrote ranked action queue to: {csv_export_path} ({len(queue_df):,} rows)")

# Export Summary JSON for Web Showcase
playbook_summary = {
    "total_pages_evaluated": len(queue_df),
    "p1_critical_count": int((queue_df['impact_tier'] == 'P1_CRITICAL').sum()),
    "p2_high_count": int((queue_df['impact_tier'] == 'P2_HIGH').sum()),
    "p3_medium_count": int((queue_df['impact_tier'] == 'P3_MEDIUM').sum()),
    "p4_low_count": int((queue_df['impact_tier'] == 'P4_LOW').sum()),
    "action_distribution": queue_df['suggested_action'].value_counts().to_dict(),
    "reason_code_counts": queue_df['reason_codes'].value_counts().head(10).to_dict(),
    "intended_use": intended_use_spec,
    "review_protocol": review_protocol,
    "monitoring_config": monitoring_config
}

json_export_path = out_dir / 'action_playbook_summary.json'
with open(json_export_path, 'w') as f:
    json.dump(playbook_summary, f, indent=2)
print(f"Wrote playbook summary JSON to: {json_export_path}")


Wrote ranked action queue to: work\outputs\action_playbook_queue.csv (30,000 rows)
Wrote playbook summary JSON to: work\outputs\action_playbook_summary.json


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.